# 🧱 Lab: Fault Injection Hardening

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a small processing pipeline that validates inputs, retries on transient errors, and logs every step
2. **Inject** realistic faults (500s, timeouts, malformed JSON, empty rows) into the pipeline using a mock API
3. **Verify** that the pipeline handles each fault gracefully — no silent failures, no infinite retries, no leaked secrets
4. **Apply** a debugging decision procedure to diagnose which layer caught (or missed) each injected fault

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Boundary Validation | Check user input and model output, the parts you do not control, so errors, abuse, and surprises are caught before they hit your core logic. | mini-boundary-schemas |
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. | mini-settings-from-env |
| Narrow Exception Handling | Custom exception types beat bare `except`, never swallow errors. Catch only what you can handle, add context, then raise or crash so failures are visible and fixable. | mini-narrow-exceptions |
| Bounded Retry | Retry transient errors only, and cap attempts, total wall-clock time, and cost or rate-limit budget so failures end fast, predictably, and safely. | mini-bounded-retry |
| Structured Logging | Logs as records with a run id and consistent per-record fields, so you can trace behavior and search reliably, while redacting sensitive keys and user text by default. | mini-structured-logging |
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | This lab |

## Prerequisites

- **mini-boundary-schemas**: Validating inputs/outputs with dataclass schemas at system boundaries
- **mini-settings-from-env**: Building layered configuration from defaults, `.env`, and environment overrides
- **mini-narrow-exceptions**: Replacing bare `except` with narrow custom exceptions that fail loudly
- **mini-bounded-retry**: Implementing capped retries with backoff, jitter, and a time budget
- **mini-structured-logging**: Emitting JSON logs with run IDs and redacting secrets


## The Scenario

You have a tiny **document-summarization pipeline**. It reads a batch of document records, validates each one, calls a "summarization API" (mocked), validates the response, and logs every outcome.

Your job: wire together every hardening layer from the module, then **break it on purpose** with four injected faults and confirm the pipeline survives each one gracefully.

| Fault | What it simulates | Expected behaviour |
|-------|-------------------|--------------------|
| **HTTP 500** | Server crash | Retry with backoff, then skip the item |
| **Timeout** | Slow/hung server | Retry with backoff, then skip the item |
| **Malformed JSON** | Garbled API response | Output validation catches it, skip the item |
| **Empty row** | Missing data in the batch | Input validation catches it before any API call |

## Step 1 — Configuration and Logging Setup

We start by building the two foundational layers: **layered configuration** (from `mini-settings-from-env`) and **structured logging** with redaction (from `mini-structured-logging`). These must exist before any processing code runs.

In [1]:
import json
import logging
import os
import random
import re
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from IPython.display import display, Markdown


def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# ── Layered Configuration ──────────────────────────────────────
# Defaults → environment overrides (no real .env needed for this lab)

DEFAULTS = {
    "MAX_RETRIES": 3,
    "RETRY_BASE_DELAY": 0.1,   # small for demo speed
    "RETRY_BUDGET_SEC": 5.0,
    "API_KEY": "sk-test-not-a-real-key-1234567890",
    "MODEL": "gpt-4o-mini",
}


def get_setting(name: str) -> str:
    """Return env override if set, else the default."""
    return os.environ.get(name, DEFAULTS[name])


# ── Structured Logger with Redaction ──────────────────────────

class JSONFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        entry = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "msg": record.getMessage(),
        }
        for key in ("run_id", "item_id", "fault", "attempt",
                    "latency_ms", "status", "error"):
            if hasattr(record, key):
                entry[key] = getattr(record, key)
        return json.dumps(entry)


class RedactingFilter(logging.Filter):
    KEY_PATTERN = re.compile(r"(sk-|key-|token-)[A-Za-z0-9_-]{4,}")

    def filter(self, record: logging.LogRecord) -> bool:
        record.msg = self.KEY_PATTERN.sub(
            lambda m: m.group()[:6] + "...REDACTED", str(record.msg)
        )
        return True


def get_logger(name: str = "pipeline") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(JSONFormatter())
        logger.addHandler(handler)
        logger.addFilter(RedactingFilter())
        logger.setLevel(logging.DEBUG)
    return logger


RUN_ID = uuid.uuid4().hex[:8]
log = get_logger()

log.info(
    f"Pipeline configured — key={get_setting('API_KEY')}",
    extra={"run_id": RUN_ID},
)
print(f"\nRun ID: {RUN_ID}")
print(f"Max retries: {get_setting('MAX_RETRIES')}")
print(f"Budget: {get_setting('RETRY_BUDGET_SEC')}s")

{"ts": "2026-09-17T01:19:05.761864+00:00", "level": "INFO", "msg": "Pipeline configured \u2014 key=sk-tes...REDACTED", "run_id": "6d8af610"}



Run ID: 6d8af610
Max retries: 3
Budget: 5.0s


Notice the API key was redacted in the log line — `sk-test...REDACTED` — even though we passed the full key. The redaction filter is working.

## Step 2 — Boundary Schemas (Input + Output Validation)

We define **two boundary schemas** — one for the incoming document records and one for the API response. Any data that crosses these boundaries gets validated before we trust it.

In [2]:
# ── Custom Exceptions ─────────────────────────────────────────

class InputValidationError(ValueError):
    """Raised when an input record fails boundary validation."""
    def __init__(self, item_id: str, reason: str):
        self.item_id = item_id
        self.reason = reason
        super().__init__(f"Input invalid [{item_id}]: {reason}")


class OutputValidationError(ValueError):
    """Raised when an API response fails boundary validation."""
    def __init__(self, item_id: str, reason: str):
        self.item_id = item_id
        self.reason = reason
        super().__init__(f"Output invalid [{item_id}]: {reason}")


class TransientAPIError(Exception):
    """Retryable server failure (500, 502, 503, timeout)."""
    def __init__(self, message: str, status_code: int | None = None):
        self.status_code = status_code
        super().__init__(message)


class PermanentAPIError(Exception):
    """Non-retryable failure (400, 401)."""
    def __init__(self, message: str, status_code: int | None = None):
        self.status_code = status_code
        super().__init__(message)


class BudgetExhaustedError(Exception):
    """Total time budget for retries exceeded."""


# ── Boundary Schemas ──────────────────────────────────────────

@dataclass
class DocInput:
    doc_id: str
    text: str


@dataclass
class SummaryOutput:
    doc_id: str
    summary: str


def validate_input(raw: dict) -> DocInput:
    """Validate a raw dict at the input boundary."""
    doc_id = raw.get("doc_id", "unknown")
    if not isinstance(raw.get("doc_id"), str) or not raw["doc_id"].strip():
        raise InputValidationError(doc_id, "missing or empty doc_id")
    if not isinstance(raw.get("text"), str) or not raw["text"].strip():
        raise InputValidationError(doc_id, "missing or empty text")
    return DocInput(doc_id=raw["doc_id"].strip(), text=raw["text"].strip())


def validate_output(raw: dict, expected_id: str) -> SummaryOutput:
    """Validate the API response at the output boundary."""
    if not isinstance(raw, dict):
        raise OutputValidationError(expected_id, f"expected dict, got {type(raw).__name__}")
    if not isinstance(raw.get("summary"), str) or not raw["summary"].strip():
        raise OutputValidationError(expected_id, "missing or empty summary")
    return SummaryOutput(doc_id=expected_id, summary=raw["summary"].strip())


# Quick smoke test
good = validate_input({"doc_id": "doc-001", "text": "Quarterly earnings grew 12%."})
print(f"✅ Input validated: {good}")

good_out = validate_output({"summary": "Earnings up 12%."}, "doc-001")
print(f"✅ Output validated: {good_out}")

✅ Input validated: DocInput(doc_id='doc-001', text='Quarterly earnings grew 12%.')
✅ Output validated: SummaryOutput(doc_id='doc-001', summary='Earnings up 12%.')


## Step 3 — Fault-Injecting Mock API + Bounded Retry

This is the core of the lab. We build a mock API that can be **scripted** to return different faults per document, plus the bounded retry wrapper from `mini-bounded-retry`.

In [3]:
class FaultingMockAPI:
    """
    A mock summarization API with per-document fault injection.

    fault_map: {doc_id: [list of responses in order]}
    Each response is one of:
      - {"status": 200, "body": {"summary": "..."}}
      - {"status": 500}
      - {"status": "timeout"}
      - {"status": 200, "body": "not-json"}        (malformed)
    """

    def __init__(self, fault_map: dict):
        # deep copy so we can pop without mutating the original
        self.fault_map = {k: list(v) for k, v in fault_map.items()}
        self.call_log = []  # record every call for inspection

    def summarize(self, doc: DocInput) -> dict:
        """Simulate a summarization API call."""
        responses = self.fault_map.get(doc.doc_id, [])

        if responses:
            resp = responses.pop(0)
        else:
            # Default: succeed
            resp = {"status": 200, "body": {"summary": f"Summary of {doc.doc_id}."}}

        self.call_log.append({"doc_id": doc.doc_id, "response": resp})

        status = resp["status"]

        if status == "timeout":
            raise TransientAPIError("Request timed out", status_code=408)
        elif status in (500, 502, 503):
            raise TransientAPIError(f"Server error {status}", status_code=status)
        elif status == 200:
            return resp["body"]  # might be malformed!
        else:
            raise PermanentAPIError(f"Client error {status}", status_code=status)


# ── Bounded Retry (from mini-bounded-retry) ────────────────────

def retry_with_budget(fn, max_attempts=3, base_delay=0.1, budget_seconds=5.0):
    """Call fn() with bounded retries on TransientAPIError."""
    start = time.monotonic()

    for attempt in range(1, max_attempts + 1):
        elapsed = time.monotonic() - start
        if elapsed >= budget_seconds:
            raise BudgetExhaustedError(
                f"Budget of {budget_seconds}s exhausted after {attempt - 1} attempts"
            )
        try:
            return fn()
        except PermanentAPIError:
            raise
        except TransientAPIError as exc:
            if attempt == max_attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            jitter = random.uniform(0, delay * 0.5)
            wait = min(delay + jitter, budget_seconds - (time.monotonic() - start))
            wait = max(wait, 0)
            log.warning(
                f"Attempt {attempt}/{max_attempts} failed: {exc} → retry in {wait:.2f}s",
                extra={"run_id": RUN_ID, "attempt": attempt, "fault": str(exc)},
            )
            time.sleep(wait)


print("Mock API and retry logic ready.")

Mock API and retry logic ready.


## Step 4 — The Processing Pipeline

This function ties every layer together: **input validation → retry-wrapped API call → output validation → structured logging**. Each layer catches its specific failure type. Nothing is silently swallowed.

In [4]:
def process_batch(records: list[dict], api: FaultingMockAPI) -> dict:
    """
    Process a batch of document records through the full pipeline.
    Returns a report dict with successes, failures, and reasons.
    """
    max_retries = int(get_setting("MAX_RETRIES"))
    base_delay = float(get_setting("RETRY_BASE_DELAY"))
    budget = float(get_setting("RETRY_BUDGET_SEC"))

    results = {"success": [], "failed": []}

    for i, raw in enumerate(records):
        doc_id = raw.get("doc_id", f"row-{i}")

        # ── Layer 1: Input Validation ──────────────────────────
        try:
            doc = validate_input(raw)
        except InputValidationError as exc:
            log.error(
                f"INPUT_INVALID: {exc}",
                extra={"run_id": RUN_ID, "item_id": doc_id, "fault": "input_validation"},
            )
            results["failed"].append({"doc_id": doc_id, "reason": str(exc)})
            continue

        # ── Layer 2: API Call with Bounded Retry ───────────────
        try:
            raw_response = retry_with_budget(
                lambda d=doc: api.summarize(d),
                max_attempts=max_retries,
                base_delay=base_delay,
                budget_seconds=budget,
            )
        except (TransientAPIError, BudgetExhaustedError) as exc:
            log.error(
                f"API_FAILED: {exc}",
                extra={"run_id": RUN_ID, "item_id": doc_id, "fault": "api_transient"},
            )
            results["failed"].append({"doc_id": doc_id, "reason": str(exc)})
            continue
        except PermanentAPIError as exc:
            log.error(
                f"API_PERMANENT: {exc}",
                extra={"run_id": RUN_ID, "item_id": doc_id, "fault": "api_permanent"},
            )
            results["failed"].append({"doc_id": doc_id, "reason": str(exc)})
            continue

        # ── Layer 3: Output Validation ─────────────────────────
        try:
            summary = validate_output(raw_response, doc.doc_id)
        except OutputValidationError as exc:
            log.error(
                f"OUTPUT_INVALID: {exc}",
                extra={"run_id": RUN_ID, "item_id": doc_id, "fault": "output_validation"},
            )
            results["failed"].append({"doc_id": doc_id, "reason": str(exc)})
            continue

        # ── Success ────────────────────────────────────────────
        log.info(
            f"OK: {doc.doc_id}",
            extra={"run_id": RUN_ID, "item_id": doc_id, "status": "success"},
        )
        results["success"].append({"doc_id": doc.doc_id, "summary": summary.summary})

    return results


print("process_batch() defined — all four layers wired together.")

process_batch() defined — all four layers wired together.


## Step 5 — Inject Faults and Run the Pipeline

Now the fun part: we create a batch with **four documents**, each designed to trigger a different fault. A fifth document is clean to confirm the happy path still works.

In [5]:
# ── Test Batch ────────────────────────────────────────────────
batch = [
    # 1. Clean document — should succeed
    {"doc_id": "doc-ok", "text": "Revenue increased 15% year over year."},

    # 2. Empty row — input validation should catch this
    {"doc_id": "doc-empty", "text": ""},

    # 3. Server 500s then recovers — retry should handle it
    {"doc_id": "doc-500", "text": "The board approved a new strategic plan."},

    # 4. Timeout then recovers — retry should handle it
    {"doc_id": "doc-timeout", "text": "Customer satisfaction scores improved."},

    # 5. Malformed JSON response — output validation should catch it
    {"doc_id": "doc-bad-json", "text": "New partnerships were announced."},
]

# ── Fault Script ──────────────────────────────────────────────
fault_map = {
    # doc-ok:       no faults → default success
    # doc-empty:    never reaches the API (input validation)
    "doc-500":     [
        {"status": 500},
        {"status": 500},
        {"status": 200, "body": {"summary": "Board approved new strategy."}},
    ],
    "doc-timeout":  [
        {"status": "timeout"},
        {"status": 200, "body": {"summary": "Customer satisfaction improved."}},
    ],
    "doc-bad-json": [
        {"status": 200, "body": "<<not valid json output>>"},  # string, not dict
    ],
}

api = FaultingMockAPI(fault_map)

print("Batch: 5 documents, 4 distinct fault scenarios")
print("Running pipeline...\n")
print("=" * 72)

report = process_batch(batch, api)

print("=" * 72)

{"ts": "2026-09-17T01:19:05.969714+00:00", "level": "INFO", "msg": "OK: doc-ok", "run_id": "6d8af610", "item_id": "doc-ok", "status": "success"}


{"ts": "2026-09-17T01:19:05.971732+00:00", "level": "ERROR", "msg": "INPUT_INVALID: Input invalid [doc-empty]: missing or empty text", "run_id": "6d8af610", "item_id": "doc-empty", "fault": "input_validation"}


{"ts": "2026-09-17T01:19:05.971732+00:00", "level": "WARNING", "msg": "Attempt 1/3 failed: Server error 500 \u2192 retry in 0.15s", "run_id": "6d8af610", "fault": "Server error 500", "attempt": 1}


Batch: 5 documents, 4 distinct fault scenarios
Running pipeline...



{"ts": "2026-09-17T01:19:06.119884+00:00", "level": "WARNING", "msg": "Attempt 2/3 failed: Server error 500 \u2192 retry in 0.26s", "run_id": "6d8af610", "fault": "Server error 500", "attempt": 2}


{"ts": "2026-09-17T01:19:06.385449+00:00", "level": "INFO", "msg": "OK: doc-500", "run_id": "6d8af610", "item_id": "doc-500", "status": "success"}


{"ts": "2026-09-17T01:19:06.385449+00:00", "level": "WARNING", "msg": "Attempt 1/3 failed: Request timed out \u2192 retry in 0.11s", "run_id": "6d8af610", "fault": "Request timed out", "attempt": 1}


{"ts": "2026-09-17T01:19:06.497984+00:00", "level": "INFO", "msg": "OK: doc-timeout", "run_id": "6d8af610", "item_id": "doc-timeout", "status": "success"}


{"ts": "2026-09-17T01:19:06.499631+00:00", "level": "ERROR", "msg": "OUTPUT_INVALID: Output invalid [doc-bad-json]: expected dict, got str", "run_id": "6d8af610", "item_id": "doc-bad-json", "fault": "output_validation"}


## Step 6 — Inspect the Results

Let's display the report as a readable table and then apply the **debugging decision procedure**: for each failure, which hardening layer caught it?

In [6]:
# ── Successes ─────────────────────────────────────────────────
lines = ["## Pipeline Report\n"]
lines.append(f"**Run ID:** `{RUN_ID}`\n")
lines.append(f"### ✅ Successes ({len(report['success'])})\n")
lines.append("| Doc ID | Summary |")
lines.append("|--------|---------|")
for s in report["success"]:
    lines.append(f"| {s['doc_id']} | {s['summary']} |")

lines.append(f"\n### ❌ Failures ({len(report['failed'])})\n")
lines.append("| Doc ID | Reason |")
lines.append("|--------|--------|")
for f in report["failed"]:
    lines.append(f"| {f['doc_id']} | {f['reason']} |")

md("\n".join(lines))

## Pipeline Report

**Run ID:** `6d8af610`

### ✅ Successes (3)

| Doc ID | Summary |
|--------|---------|
| doc-ok | Summary of doc-ok. |
| doc-500 | Board approved new strategy. |
| doc-timeout | Customer satisfaction improved. |

### ❌ Failures (2)

| Doc ID | Reason |
|--------|--------|
| doc-empty | Input invalid [doc-empty]: missing or empty text |
| doc-bad-json | Output invalid [doc-bad-json]: expected dict, got str |

## Step 7 — Debugging Decision Procedure

When a fault happens in production, you need a **systematic procedure** to figure out which layer caught it — or whether it slipped through. Let's verify our pipeline against each injected fault.

In [7]:
# ── Verify: which layer caught each fault? ────────────────────

expected_outcomes = [
    {
        "doc_id": "doc-ok",
        "fault": "None (clean)",
        "expected_layer": "—",
        "expected_result": "success",
    },
    {
        "doc_id": "doc-empty",
        "fault": "Empty text",
        "expected_layer": "Input Validation",
        "expected_result": "failed",
    },
    {
        "doc_id": "doc-500",
        "fault": "HTTP 500 × 2, then 200",
        "expected_layer": "Bounded Retry",
        "expected_result": "success",
    },
    {
        "doc_id": "doc-timeout",
        "fault": "Timeout × 1, then 200",
        "expected_layer": "Bounded Retry",
        "expected_result": "success",
    },
    {
        "doc_id": "doc-bad-json",
        "fault": "Malformed response",
        "expected_layer": "Output Validation",
        "expected_result": "failed",
    },
]

# Check actual results
success_ids = {s["doc_id"] for s in report["success"]}
failed_ids = {f["doc_id"] for f in report["failed"]}

lines = ["## Debugging Decision Procedure\n"]
lines.append("| Doc ID | Injected Fault | Caught By | Expected | Actual | ✓ |")
lines.append("|--------|---------------|-----------|----------|--------|---|")

all_pass = True
for exp in expected_outcomes:
    actual = "success" if exp["doc_id"] in success_ids else "failed"
    ok = "✅" if actual == exp["expected_result"] else "❌"
    if actual != exp["expected_result"]:
        all_pass = False
    lines.append(
        f"| {exp['doc_id']} | {exp['fault']} | {exp['expected_layer']} "
        f"| {exp['expected_result']} | {actual} | {ok} |"
    )

lines.append(f"\n**All checks passed: {'✅ Yes' if all_pass else '❌ No'}**")

md("\n".join(lines))

## Debugging Decision Procedure

| Doc ID | Injected Fault | Caught By | Expected | Actual | ✓ |
|--------|---------------|-----------|----------|--------|---|
| doc-ok | None (clean) | — | success | success | ✅ |
| doc-empty | Empty text | Input Validation | failed | failed | ✅ |
| doc-500 | HTTP 500 × 2, then 200 | Bounded Retry | success | success | ✅ |
| doc-timeout | Timeout × 1, then 200 | Bounded Retry | success | success | ✅ |
| doc-bad-json | Malformed response | Output Validation | failed | failed | ✅ |

**All checks passed: ✅ Yes**

Every fault was caught by the correct layer. No silent failures, no infinite retries, no unvalidated data slipping through.

## Step 8 — Bonus Scenario: All Retries Exhausted

What if the server *never* recovers? The bounded retry should give up after `max_attempts` and the pipeline should log the failure cleanly — not hang.

In [8]:
# Server returns 500 every time — retries exhausted
fault_map_always_fail = {
    "doc-doomed": [{"status": 500}, {"status": 500}, {"status": 500}],
}
api_fail = FaultingMockAPI(fault_map_always_fail)

batch_fail = [
    {"doc_id": "doc-doomed", "text": "This document will never be summarized."},
]

print("Running pipeline with a permanently failing server...\n")
result_fail = process_batch(batch_fail, api_fail)

print(f"\nSuccesses: {len(result_fail['success'])}")
print(f"Failures:  {len(result_fail['failed'])}")
print(f"Reason:    {result_fail['failed'][0]['reason']}")
print(f"API calls: {len(api_fail.call_log)}  (capped at max_retries={get_setting('MAX_RETRIES')})")

{"ts": "2026-09-17T01:19:06.561102+00:00", "level": "WARNING", "msg": "Attempt 1/3 failed: Server error 500 \u2192 retry in 0.13s", "run_id": "6d8af610", "fault": "Server error 500", "attempt": 1}


{"ts": "2026-09-17T01:19:06.693251+00:00", "level": "WARNING", "msg": "Attempt 2/3 failed: Server error 500 \u2192 retry in 0.24s", "run_id": "6d8af610", "fault": "Server error 500", "attempt": 2}


Running pipeline with a permanently failing server...



{"ts": "2026-09-17T01:19:06.933907+00:00", "level": "ERROR", "msg": "API_FAILED: Server error 500", "run_id": "6d8af610", "item_id": "doc-doomed", "fault": "api_transient"}



Successes: 0
Failures:  1
Reason:    Server error 500
API calls: 3  (capped at max_retries=3)


The pipeline made exactly 3 API calls (the max), logged each retry attempt, and then recorded a clean failure. No hang, no crash, no silent zero.

## Step 9 — The Hardening Layers at a Glance

## Hardening Layer Map

<div style='display:grid;grid-template-columns:max-content max-content;align-items:center;column-gap:12px;row-gap:8px;margin:12px 0 16px 0;'>
<span style='grid-column:1;grid-row:1;justify-self:stretch;border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>Raw Input</strong><br>dict</span>
<span style='grid-column:1;grid-row:2;justify-self:center;'>↓</span>
<span style='grid-column:1;grid-row:3;justify-self:stretch;border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>Input Validation</strong><br>boundary schema</span>
<span style='grid-column:2;grid-row:3;'>catches empty rows, missing fields</span>
<span style='grid-column:1;grid-row:4;justify-self:center;'>↓</span>
<span style='grid-column:1;grid-row:5;justify-self:stretch;border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>Bounded Retry</strong><br>backoff + budget</span>
<span style='grid-column:2;grid-row:5;'>catches 500s, timeouts (transient)<br>lets 401/400 through (permanent)</span>
<span style='grid-column:1;grid-row:6;justify-self:center;'>↓</span>
<span style='grid-column:1;grid-row:7;justify-self:stretch;border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>Output Validation</strong><br>boundary schema</span>
<span style='grid-column:2;grid-row:7;'>catches malformed API responses</span>
<span style='grid-column:1;grid-row:8;justify-self:center;'>↓</span>
<span style='grid-column:1;grid-row:9;justify-self:stretch;border:1px solid #888;border-radius:6px;padding:8px 12px;text-align:center;'><strong>Success</strong></span>
</div>

All layers log via **structured logging** with run_id + redaction.
All layers use **narrow exceptions** — no bare except anywhere.
All settings come from **layered configuration**.


## 🎯 Summary

### What You Built

You built a fault-tolerant document-processing pipeline and systematically broke it with four injected faults — HTTP 500s, timeouts, malformed JSON responses, and empty input rows — verifying that every hardening layer caught its designated failure without silent errors or infinite retries.

### Key Takeaways

1. **Boundary validation** — checking input and output schemas at system edges catches bad data before it corrupts downstream logic (empty rows never reach the API; garbled responses never reach your application)
2. **Bounded retry with backoff** — transient server errors (500s, timeouts) recover automatically, while max-attempts and a time budget prevent the pipeline from hanging
3. **Narrow exceptions** — each layer catches only the errors it can handle (`InputValidationError`, `TransientAPIError`, `OutputValidationError`), letting bugs propagate loudly
4. **Structured logging with redaction** — every decision (retry, skip, success) is logged as a JSON record with a `run_id`, making post-mortem debugging a filter query instead of a guessing game
5. **Debugging decision procedure** — after injecting faults, systematically mapping each failure to the layer that caught it confirms your hardening has no gaps